# mesh01 — 표적 10종을 무엇으로 짓고, 무엇으로 채점하나 (전체 지도)

> ⚠ **이 노트북은 생성물이다.** 수정은 `report_mesh/src/make_mesh01.py` 에서 하고
> 재실행할 것(`.ipynb` 를 직접 고치면 다음 빌드에서 사라진다).

**이 편이 답하는 질문** — 우리가 레이더 시뮬레이션에 쓰는 드론 표적 10종은 무엇으로 만들어졌고, 그 형상을 무엇이 채점하는가.

**무엇을 근거로 하는가** (본문 수치는 손으로 적지 않고 아래 **원장** — 검사기가 낸 측정 기록 파일 — 에서 주입한다)

| 원장 | 무엇이 들어 있나 |
|---|---|
| `outputs/mesh_inspect_body_arms_0816.json` | 동체·팔·다리·모터 벨 전수 실측(10종) + matrice4e 공식 CAD 정정 착지 검증 |
| `outputs/mesh_inspect_gimbal_sensors_0816.json` | 짐벌·카메라·센서 검사(10종) — 부착 게이트 A~D |
| `outputs/mesh_inspect_materials_check_0816.json` | 재질 배정·검사기·프롭 외 부품 감사(13그룹·10종) |
| `outputs/meshfix_matrice4e.json` | DJI 공식 STEP 대조 정정 명세 14건(matrice4e) |
| `docs/MESH_AUDIT_0816.md` | 적대적 감사 — 발견·반증·수리 우선순위(§⑤) |
| `assets/meshes/reference/SOURCES.md` | 참조 CAD·스캔의 출처와 라이선스 |
| `report_mesh/outputs/mesh_verify_canon_0817.json` | ⭐**정본 판** 기하 검증 A·B·C·D·F·G — 이 편 수치의 기본 원장 |
| `report_mesh/outputs/mesh_canon_0817.json` | ⭐**정본 판** 스위치·예산 스냅샷·재질 가중 매몰면 |
| `docs/MESH_CERTIFICATE.md` | 메쉬 인증서 — 무엇을 장담하고 무엇은 못 하는가(판정 «조건부 장담») |
| `report_mesh/outputs/mesh_verify.json` | 기하 검증 스위트 A~I — H·I(GPU 절)는 아직 이쪽만 있다 |

**한 줄 요약** — 인터넷의 드론 3D 모델은 시각용 껍데기라 레이더 시뮬레이션에 못 쓴다.
그래서 제조사 공식 제원과 (있는 경우) 공식 CAD 로부터 코드가 드론 10종을 직접 깎는다
(총 301,506개 삼각형 · 부품 302개). 이 편은 그 **전체 지도**다 —
자료가 어디까지 제작에 들어갔고, 어디서부터 독립 채점이 시작되는지.

⭐ **이 편이 설명하는 메쉬는 «정본 판»이다** — 파일명 꼬리표 `_mfixbatteryi5_blperairframe`.
어떤 스위치가 기본으로 켜져 있고 옛 판을 어떻게 되살리는지는 §4.1 에 있다.

이 시리즈(mesh01~08)는 **파이썬 기초만 아는 독자**를 위한 3D 모델 제작 가이드다.
본문의 모든 숫자는 손으로 적지 않고 위 원장에서 자동 주입했으며,
**모든 사실 옆에 `← 출처:` 를 단다** — 어느 파일·어느 측정에서 온 정보인지 추적할 수 있게.

## 용어풀이 (이 리포트에 나오는 순서대로)

| 용어 | 한 줄 풀이 |
|---|---|
| **메쉬(mesh)** | 작은 삼각형 수천 장을 이어붙여 만든 3D 표면. 컴퓨터가 형상을 저장하는 가장 보편적 방식 |
| **꼭짓점(vertex)** | 3D 공간의 점 (x, y, z). 삼각형의 모서리 끝점 |
| **면(face)** | 꼭짓점 3개를 이은 삼각형 1장. 메쉬의 최소 단위 |
| **법선(normal)** | 삼각형이 바라보는 방향(수직 화살표). 물체의 '겉'과 '속'을 구분한다 |
| **수밀(watertight)** | 구멍이 하나도 없는 닫힌 표면 — 물을 부어도 새지 않는 그릇 |
| **경계 모서리** | 삼각형 **하나만** 쓰는 모서리. 구멍의 테두리다 |
| **그룹(group)** | 면마다 붙인 부위 이름표(body/prop/motor …). 부위별 재질 배정의 열쇠 |
| **OBJ** | 꼭짓점·삼각형 목록을 적는 텍스트 3D 파일 형식. Sionna/Mitsuba 가 바로 읽는다 |
| **파라메트릭 CAD** | 치수(파라미터)를 넣으면 코드가 형상을 만들어 주는 설계 방식 |
| **불리언(boolean union)** | 두 입체를 '합집합' 으로 녹여 붙여 내부의 숨은 면을 없애는 연산 |
| **축간거리(wheelbase)** | 마주 보는 두 로터 축 사이의 거리. **메쉬에서 잰 값** |
| **대각(diagonal)** | 제조사가 공표하는 대각 치수. 축간거리와 **같지 않을 수 있다**(§1.3) |
| **RCS(σ)** | 레이더에게 물체가 얼마나 '밝게' 보이는지의 면적값 [m²] |
| **PO / SBR** | 물리광학 / 광선발사·반사 — 표면에서 RCS 를 계산하는 두 방법 |
| **\|Γ\|(반사계수)** | 전파가 재질 표면에서 반사되는 진폭 비율. 금속≈1.0, 플라스틱≈0.3 |
| **평판극한 상한** | 형상 차이를 «같은 크기 평판이면 최대 이만큼» 으로 옮긴 **상한값**. 커널 계산 결과가 아니라 크기 감각을 주는 자다 |
| **파장 λ** | 전파의 한 주기 길이. 메쉬 삼각형은 λ 보다 충분히 작아야 형상을 '전파의 눈'으로 담는다 |
| **정본(canon) 판** | 지금 기본으로 지어지는 형상. 스위치를 아무것도 안 주면 이 판이 나온다(§4.1) |
| **파일명 꼬리표** | 산출물 이름 끝에 붙는 표시. 어느 판에서 계산한 결과인지 이름만으로 갈린다 |
| **예산(budget)** | 검사가 «이만큼까지는 통과» 로 선언해 둔 값. «옳다» 가 아니라 «지금 이만큼이다» 라는 뜻(§4.3) |
| **매몰면(buried face)** | 다른 부품 **속**에 들어가 실물이라면 안 보이는 삼각형. PO 는 가림을 안 보므로 그 면적을 두 번 센다 |
| **골든 봉인** | 오늘 형상의 지문을 떠 두고 내일과 대조하는 장치. 지키는 것은 «안 바뀜» 이지 «옳음» 이 아니다 |

## 0. 이 편이 답하는 질문

이 저장소는 통신 신호를 조명 삼아 드론을 탐지하는 시뮬레이터다. 시나리오는 하나가 아니다 —
**패시브 바이스태틱**(남의 신호를 빌려 쓴다)과 **모노스태틱**(파형을 우리가 안다)을 함께 본다
← 출처: `README.md`. 그 시뮬레이션의 **표적**이 되는
드론 10종(Mini 5 Pro, Mavic 4 Pro, Matrice 4E, S1000+, Phantom 4, Typhoon H (H480), X500 V2, Phantom 3 Professional, Matrice 350 RTK, Mini 2)의
3D 모델을 어떻게 만들었는지가 이 시리즈의 주제다.

⭐ **표적 축은 시나리오와 무관하다.** σ·마이크로도플러·앙각은 «누가 신호를 쐈나» 와 상관없이
같은 형상에서 나온다. 그래서 이 시리즈는 시나리오를 안 고르고 형상만 다룬다.

이 편(mesh01)은 다섯 가지 질문에 답한다:

1. **삼각형 메쉬가 뭔가?** — 종이접기 비유로 (§1)
2. **왜 인터넷 3D 모델을 안 받고 코드로 만들었나?** (§2)
3. **그럼 참조 자료는 어디까지 제작에 들어갔나?** (§2.3 — 이 편에서 가장 자주 오해받는 자리)
4. **'OBJ 1개 = 부위 1개 = 재질 1개' 는 무슨 뜻인가?** (§3)
5. **자료 → 제작 → 검사기 → 원장, 전체 지도는 어떻게 생겼나?** (§4 — 지금 기본으로
   켜져 있는 판은 §4.1, 무엇을 장담하고 무엇은 못 하는지는 §4.5)

세부(몸체 깎는 법, 프로펠러, 자료 출처, 재질, 기하 품질, 실물 대조)는
mesh02~08 각 편이 하나씩 맡는다 — 목차는 §5.

## 1. 삼각형 메쉬란 무엇인가 — 종이접기 다면체

**비유**: 축구공 모양 종이 다면체를 접어 본 적이 있는가? 평평한 종이 조각(오각형·육각형)을
수십 장 이어붙이면 거의 둥근 공이 된다. 조각을 잘게 쓸수록 더 매끈해진다.
3D 메쉬가 정확히 그것이다 — 다만 조각이 전부 **삼각형**이고, 종이 대신 숫자로 접는다.

왜 하필 삼각형인가? **점 3개는 반드시 한 평면 위에 있기 때문**이다. 사각형부터는 뒤틀릴 수
있어(네 점이 한 평면에 안 놓임) 계산이 모호해진다. 그래서 그래픽스·전파 시뮬레이션 모두
삼각형을 최소 단위로 쓴다.

우리 코드의 메쉬 정의는 놀랄 만큼 단순하다:

```
핵심 개념 — 메쉬(Mesh)는 딱 두 가지로 이루어집니다
  1) 꼭짓점(vertex) 목록 : 3D 점 (x, y, z) 들의 리스트
  2) 면(face) 목록       : "몇 번 꼭짓점 3개를 이어 삼각형을 만들지"
거기에 우리는 "그룹(group)" 하나를 더 붙입니다. 면마다 어떤 **재질 그룹**
(예: body / arm / motor / prop / absorber ...)에 속하는지 이름표를 답니다.
```

← 출처: `src/geom.py` 모듈 docstring (그대로 인용).
실제 클래스도 딱 세 줄이다 — `.v`(꼭짓점), `.f`(삼각형 인덱스), `.g`(면별 그룹 이름)
← 출처: `src/geom.py` `class Mesh` docstring.

### 1.2 법선과 watertight — '겉면 스티커'와 '물 안 새는 그릇'

종이 다면체를 접을 때 겉과 속을 뒤집어 붙이면 이상해진다. 메쉬도 같다. 삼각형마다
**법선(normal)** — 어느 쪽이 '겉'인지 가리키는 수직 화살표 — 이 있고, 모든 법선이
바깥을 향해야 전파 시뮬레이터가 "여기가 물체 표면" 을 올바로 인식한다.
우리 PO 커널의 조명 판정이 `n̂·û > 0` 이라, 법선이 뒤집힌 면은 조명 여부가 **반대로** 정해진다
← 출처: `src/rcs_po.py` 조명 판정.

**watertight(수밀)** 는 표면에 구멍이 하나도 없다는 뜻이다 — 그릇에 물을 부어도 안 샌다.
구멍이 있으면 (a) 부피를 정의할 수 없고 (b) **안/밖 판정(`contains`)이 정의되지 않는다.**
두 번째가 실질적인 피해다: 내부 판정을 쓰는 검사가 그 부품을 조용히 건너뛴다.

두 개념을 여기서는 **뜻만** 익히고, 지금 우리 메쉬의 성적표는 §4.3~§4.4 에서
검사기·예산과 함께 읽는다 — 숫자 하나로 «통과» 를 말하는 것이 정확하지 않기 때문이다.

### 1.3 표적 10종 — 한 표로

| 드론 | 근거 | 축간거리[mm] | 프롭Ø[mm]×수 | 꼭짓점 | 삼각형 | 그룹 | 수밀 | 지금 선언된 결함 |
|---|---|---|---|---|---|---|---|---|
| Mini 5 Pro | **[B]** | 248.3 | 152×4 | 14,956 | 29,824 | 9 | 22/22 | 세로 배율 1.2985(전 부품 늘림) |
| Mavic 4 Pro | **[C]** | 441.0 | 267×4 | 15,682 | 31,280 | 8 | 21/21 | 세로 배율 1.3524 · 짐벌이 발보다 15.35 mm 아래 |
| Matrice 4E | **[A]** | 438.9 | 274×4 | 15,939 | 31,766 | 9 | 28/28 | 로터면이 CAD 보다 18.5 mm 위(F19~F21 보류) |
| S1000+ | **[B]** | 1,043.5 | 381×8 | 19,019 | 37,902 | 10 | 49/49 | 카본 센터플레이트가 `plastic` 그룹 · 바깥 참값 0행 |
| Phantom 4 | **[B]** | 350.0 | 240×4 | 14,935 | 29,782 | 8 | 22/22 | 착륙아치 8.3~8.5 mm 뜸 |
| Typhoon H (H480) | **[A]** | 480.0 | 230×6 | 18,360 | 36,604 | 9 | 29/29 | — |
| X500 V2 | **[A]** | 500.0 | 254×4 | 10,199 | 20,198 | 10 | 50/50 | 레일 4.0 mm 뜸 · accent↔arm 동일평면 |
| Phantom 3 Professional | **[B]** | 350.0 | 240×4 | 14,880 | 29,676 | 8 | 21/21 | 착륙아치 13.7~13.8 mm 뜸 · 짐벌 3조각이 떨어져 있음 |
| Matrice 350 RTK | **[B]** | 895.0 | 533×4 | 13,802 | 27,432 | 9 | 43/43 | 프롭 허브가 벨 위 6.0 mm 뜸 · 바깥 참값 0행 |
| Mini 2 | **[A]** | 213.1 | 119×4 | 13,555 | 27,042 | 8 | 17/17 | — |
| **합계** | | | | **151,327** | **301,506** | | **302/302** | |

← 출처: `report_mesh/outputs/mesh_verify_canon_0817.json` `A_geometry`(면·그룹·수밀)·
`report_mesh/outputs/mesh_canon_0817.json` `per_drone.*.wheelbase_mm`(축간거리).

**«축간거리» 열을 쓰는 이유** — 이 열은 마주 보는 두 로터 축 사이 거리를 **메쉬에서 잰 값**이다.
제조사가 공표하는 «대각» 과 같지 않을 수 있다. 가장 큰 차이는 Mini 5 Pro 로,
공표 대각 275 mm ↔ 축간거리
248.26 mm 다. 로터가 정사각형이 아니라
사다리꼴로 놓이기 때문이고, 이것은 결함이 아니라 **선언된 선택**이다
← 출처: `src/drones.py` mini5pro `note`(«diagonal_mm 은 로터 위치를 정하지 않는다»).

⚠ **리포트·발표에서 Mini 5 Pro 의 로터 간격을 쓸 때는 축간거리를 인용할 것.**
«대각 275 mm» 를 로터 간격으로 쓰면
9.7 % 틀린다.

**근거 등급** — 형상이 어디서 왔는가:

| 등급 | 뜻 | 이 저장소에서의 실체 |
|---|---|---|
| **[A]** | 실물 CAD 직접 | CAD 파일을 열어 잰 값. **제조사 배포물** — DJI Matrice 4T STEP · DJI Mini 2 GLB · Holybro X500 v2 STEP — 과 **로보틱스 저장소의 실물 CAD** (Yuneec Typhoon H480, ethz-asl/rotors_simulator·Apache-2.0)를 함께 포함한다 |
| **[B]** | 사진 계측 | 제품사진·매뉴얼 도해를 픽셀로 잰 값 |
| **[C]** | 계열 유추 | 다른 기체의 실측을 크기비로 옮긴 값 |
| **[D]** | 대리 | 다른 제조사의 부품을 대신 세운 값 |

⚠ 등급은 **«공표 숫자만으로는 안 정해지는 형상»** 이 어디서 왔는가를 매긴다. 공표 제원(외형 L×W×H · 프로펠러 지름 · 대각)은 10종 공통 입력이라 이 축 밖이다 — 그 숫자들의 출처는 mesh03(자료 수집 편) 이 맡는다.

| 드론 | 등급 | 근거 |
|---|---|---|
| Mini 5 Pro | **[B]** | 제품사진 계측. ⚠ 셸 높이의 1차 출처가 없다 — 공표 91 mm 가 프롭을 포함해 셸을 구속하지 않는다 |
| Mavic 4 Pro | **[C]** | 암 폭은 **Mini 5 Pro 실측의 크기비 이전**. DJI 가 Mavic 4 Pro CAD 를 공개하지 않는다 |
| Matrice 4E | **[A]** | DJI Matrice 4T 공식 STEP 로 형상 상수 14건 정정, 전부 착지 (`meshfix_matrice4e.json` · `body_arms.meshfix_matrice4e_landed`). ⚠ 짐벌·카메라 블록만 4T↔4E 가 달라 **치수는 사진, 매다는 자리만 CAD** |
| S1000+ | **[B]** | 공표 제원(카본 튜브 25 mm) + 제품사진 |
| Phantom 4 | **[B]** | 공표 제원 + 제품사진. 실기체 3D 스캔(CC-BY)은 **채점 전용**이라 제작에 안 썼다 |
| Typhoon H (H480) | **[A]** | Yuneec 실물 CAD(ethz-asl/rotors_simulator, Apache-2.0) — 암 12.0×12.6 ↔ CAD 12.002 |
| X500 V2 | **[A]** | Holybro 공식 STEP 프레임(암 단면 16.0×16.0 mm 재현) |
| Phantom 3 Professional | **[B]** | 공표 제원 + 제품사진 |
| Matrice 350 RTK | **[B]** | 공표 제원 + 제품사진(암 튜브 22 mm) |
| Mini 2 | **[A]** | DJI 공식 GLB(WM161) 실측 — 셸 6 스테이션 중 가운데 4개가 GLB 와 0.5 % 안 |

← 출처: `outputs/mesh_inspect_body_arms_0816.json`(암 단면·셸 스테이션 실측 대조)·
`outputs/meshfix_matrice4e.json`(공식 STEP)·`assets/meshes/reference/SOURCES.md`.

⚠ **이 등급은 기체 한 대에 하나씩 붙는 «가장 좋은 근거» 다 — «이 기체는 통째로 A» 라는 뜻이 아니다.**
부품 단위로 쪼개면 빈칸이 훨씬 많다. 인증서가 그것을 칸으로 센다:
기체×부품 **120칸 중 등급이 붙은 칸은 37칸**이고,
그중 «독립 참값»(우리 수로 우리를 검증하지 않는 근거)이 있는 행은 **32개**다
← 출처: `docs/MESH_CERTIFICATE.md` §4 · `outputs/mesh_cert_matrix_0816.json` `grade_matrix`.
부품 단위 표는 인증서 §4 에 있다.

### 1.4 눈으로 보기 — 대표 두 기종

대표를 **둘** 세운다. 두 기체가 다른 것을 대표하기 때문이다.

- **Mini 5 Pro** — 실측 캠페인의 표적이다 ← 출처: `README.md` 실측 계획.
  형상 근거는 사진 계측이고, **셸 높이의 1차 출처가 없다**는 한계를 그대로 안고 있다(§4.4).
- **Matrice 4E** — 공식 CAD 대조가 끝난 기체다. DJI Matrice 4T 공식 STEP 으로
  형상 상수 14건이 반영돼 있다 — **14/14 착지**(정정 명세 대비 착지 검증).
  ← 출처: `outputs/mesh_inspect_body_arms_0816.json` `meshfix_matrice4e_landed`.

![wireframe mini5pro](outputs/figures/wireframe_mini5pro.png)

**그림 1** — Mini 5 Pro 메쉬 3면 ← 그림 생성:
`report_mesh/src/viz_mesh_reports.py` `fig_wireframes()`.

![wireframe matrice4e](outputs/figures/wireframe_matrice4e.png)

**그림 2** — Matrice 4E 메쉬 3면. 같은 코드가 스펙만 바꿔 만든 것이다.

- **왼쪽(shaded)**: 색이 곧 부위(=재질)다 — 셸(body), 프로펠러(prop), 금속 모터(motor),
  짐벌(camera).
- **가운데(wireframe)**: 삼각형 뼈대. 곡면(동체·짐벌)일수록 촘촘하다.
- **오른쪽(top view)**: 위에서 본 로터 배치.

**삼각형 크기 — 세 숫자로 읽는다.** 중앙값만 적으면 최댓값을 못 본다:

| 드론 | 근거 | 꼭짓점 | 삼각형 | 그룹 | 한 변 p50[mm] | p95[mm] | **최대[mm]** | p95/λ | **최대/λ** |
|---|---|---|---|---|---|---|---|---|---|
| Mini 5 Pro | **[B]** | 14,956 | 29,824 | 9 | 3.7 | 7.3 | 82.9 | 0.13 λ | 1.44 λ |
| Matrice 4E | **[A]** | 15,939 | 31,766 | 9 | 5.3 | 12.8 | 157.6 | 0.22 λ | 2.74 λ |

λ 는 우리가 쓰는 최고 대역(WiFi 5.21 GHz)의 파장 **57.5 mm** 다
← 출처: `report_mesh/outputs/mesh_verify_canon_0817.json` `_meta.lam_hi_mm`·`A_geometry.*.edge_mm`.

**최대값이 λ 를 넘는 것을 어떻게 읽나** — 가장 긴 모서리는 배터리·기판 같은 **평평한 상자**의
모서리다. 평면은 잘게 쪼개도 같은 평면이라 형상이 나빠지지 않고, PO 적분은 면을 λ/11 로
다시 나눠 표본을 뜬다 ← 출처: `src/rcs_po.py` `mesh_to_points`. 그래서 이 최댓값은
위상 표본 문제가 아니다.

**프로펠러는 따로 재야 한다.** 위 표에 섞으면 «최대 모서리» 가 상자 쪽에 가려진다.
프롭 그룹만 잘라 다시 재면 이렇다:

| 드론 | 프롭 삼각형 | 전체 대비 | p50[mm] | p95[mm] | 최대[mm] | 최대/λ |
|---|---|---|---|---|---|---|
| Mini 5 Pro | 14,000 | 46.9 % | 3.64 | 3.98 | 6.48 | 0.11 λ |
| Mavic 4 Pro | 13,984 | 44.7 % | 6.38 | 6.91 | 11.35 | 0.20 λ |
| Matrice 4E | 13,952 | 43.9 % | 6.65 | 7.11 | 11.65 | 0.20 λ |
| S1000+ | 28,128 | 74.2 % | 9.11 | 9.83 | 16.19 | 0.28 λ |
| Phantom 4 | 13,920 | 46.7 % | 5.83 | 6.43 | 10.20 | 0.18 λ |
| Typhoon H (H480) | 21,024 | 57.4 % | 5.50 | 5.93 | 9.78 | 0.17 λ |
| X500 V2 | 13,904 | 68.8 % | 6.17 | 6.81 | 10.80 | 0.19 λ |
| Phantom 3 Professional | 13,920 | 46.9 % | 5.83 | 6.43 | 10.20 | 0.18 λ |
| Matrice 350 RTK | 13,968 | 50.9 % | 12.96 | 13.74 | 22.67 | 0.39 λ |
| Mini 2 | 13,936 | 51.5 % | 2.89 | 3.18 | 5.06 | 0.09 λ |

← 출처: `report_mesh/outputs/mesh_verify_canon_0817.json` `prop_triangles`.
함대에서 프롭의 가장 긴 모서리도 **0.39 λ** 를 안 넘는다 —
곡률이 큰 날은 상자보다 훨씬 촘촘히 쪼개져 있다.
⚠ 이 값은 **정지 메쉬**의 것이다. 마이크로도플러가 쓰는 «시간에 따른 메쉬 열» 의 정확도는
다른 축이고, 인증서가 그것을 자기 범위 밖으로 선언한다(§4.5).

## 2. 왜 인터넷 3D 모델을 그대로 안 쓰나

가장 쉬운 길은 3D 모델 공유 사이트에서 "DJI Mavic 4" 를 검색해 받는 것이다.
그 길을 버린 이유는 둘이다.

1. **치수 미검증** — 취미 모델러가 사진을 보고 눈대중으로 만든 것이 많다. RCS 는 투영 면적과
   세부 형상에 민감해서 치수가 몇 % 틀리면 답이 몇 dB 틀어진다.
2. **라이선스 제약** — 연구 산출물에 재배포 불가·상업 불가 모델을 섞으면 재현 패키지를 공개할 수 없다.

거기에 **레이더 고유의 이유**가 하나 더 붙는다. 게임·렌더용 모델은 겉모습만 그럴듯하면 되지만,
전파의 눈에는 겉껍데기 플라스틱(\|Γ\|=0.28)보다 속의 **배터리·모터 금속**
(\|Γ\|≈1.00)이 훨씬 밝다 — 같은 넓이면 반사 전력이 약 **11 dB**
(≈13배) 차이다
← 출처: `report_mesh/outputs/mesh_verify.json` `E_materials.*.gamma_map`(원본 `src/materials.py`,
ITU-R P.2040). ⚠ 이 값들은 **재질 상수**라 형상 판과 무관하다 — 정본 전환으로 안 바뀐다.

⚠ 다만 이 문장을 **«내부 금속이 없으면 속 빈 유령»** 으로 밀어붙이면 지금 상태를 과장한다.
단서 셋을 함께 적어야 한다 — §3.2 에서 숫자로 푼다.

### 2.1 공식 CAD 는 어디까지 있나 — 세 기체는 있고 나머지는 없다

«제조사가 CAD 를 공개하지 않는다» 는 **기체마다 다르다.** 지금 저장소가 가진 것:

| 자료 | 정체 | 축척 검산 |
|---|---|---|
| `assets/meshes/reference/matrice4-M4T_v2.step` | DJI **Matrice 4T** 공식 STEP | 폭 387.501 ↔ 공표 387.5 mm |
| `assets/meshes/reference/WM161_zhankai_1k.glb` | DJI **Mini 2** 공식 3D(펼침) | 축간거리 213.05 ↔ 공표 213 mm (+0.02 %) |
| `assets/meshes/reference/x500v2-frame.step` | Holybro **X500 v2** 공식 프레임 STEP | 암 단면 16.0×16.0 mm 재현 |

← 출처: `assets/meshes/reference/SOURCES.md`·`outputs/meshfix_matrice4e.json` `scale_check`·
`outputs/mesh_inspect_body_arms_0816.json` `per_drone.mini2`.

⭐ **그런데 M4T 는 우리 표적이 아니다.** 우리 표적은 Matrice 4**E** 이고 DJI 는 4E 판 CAD 를
공개하지 않는다. 그래서 이 CAD 는 **갈라 써야 한다**
← 출처: `docs/MESH_AUDIT_0816.md` §⑧(사용자 지시로 세운 상시 규칙):

| 부품군 | 4T ↔ 4E | CAD 를 써도 되나 |
|---|---|---|
| 셸(동체)·팔·다리·모터 | 공용 | ✅ 그대로 |
| 어안·비전 센서·비콘 위치 | 공용 | ✅ 그대로 |
| RTK 안테나 | 공용 | ✅ 그대로 |
| ⚠ **짐벌·카메라 블록** | **다르다** — 탑재체가 갈리는 지점 | ⛔ **치수는 쓰지 마라.** 매다는 자리(크래들·댐핑플레이트)만 공용이라 **위치는** 써도 된다 |
| 내부 기판·배터리 | CAD 는 외장 모델이라 애초에 없다 | — |

**나머지 7종에는 제조사 공식 CAD 가 없다.** Mavic 4 Pro 도 없다 — 그래서 그 기체의 암 폭은
Mini 5 Pro 실측을 크기비로 옮긴 값이고, 표에 **[C] 계열 유추**로 적혀 있다(§1.3).

⚠ 남은 불확실: «4T 와 4E 의 기체가 공용» 이라는 것은 제원·매뉴얼 대조에서 나온 판단이고
부품 단위로 전수 대조한 것은 아니다 ← 출처: `docs/MESH_AUDIT_0816.md` §⑧ 말미.

### 2.2 그래서 코드로 만든다 — 파라메트릭 CAD 의 4가지 이점

"코드로 만든다" = 치수를 넣으면 형상이 나오는 함수를 짠다는 뜻이다(파라메트릭 CAD).
각 드론은 `DroneSpec` 이라는 데이터클래스 하나로 요약된다 — 대각거리, 무게, 프로펠러
지름·날 수, 로터 수, 공식 외형(L×W×H) … ← 출처: `src/drones.py` `class DroneSpec`.

| 이점 | 설명 | 이 저장소에서 실증된 방식 |
|---|---|---|
| **재현성** | `build_drone(spec)` 한 줄로 같은 메쉬가 다시 나온다 | 메쉬 지문(sha256(정점+삼각형))을 A/B 로 비교해 10종 **비트동일** 확인. ⭐**옛 판도 되살아난다** — `MESH_FIX=none BLADE_LAW=legacy` 를 주면 정본 이전 형상이 비트동일하게 다시 나온다(§4.1) |
| **파라메트릭** | 제원이 정정되면 숫자 하나만 고치면 형상 전체가 따라온다 | matrice4e 상수 14건 정정이 한 번에 착지 |
| **부위별 재질** | 만들 때부터 면마다 body/prop/motor … 이름표 → 재질 배정이 공짜 | §3 |
| **버전관리** | 메쉬가 곧 코드니까 형상 변경 이력이 텍스트로 남는다 | 바이너리 3D 파일로는 불가능 |

← 출처: 설계 의도는 `src/drones.py`·`src/geom.py` 모듈 docstring;
지문 A/B 검증은 `outputs/mesh_inspect_body_arms_0816.json` `code_changes.verification`.

**정직성 원칙 — 모르는 값은 모른다고 적는다.** 예컨대 Mini 5 Pro:

> Mini 5 Pro 의 대각(모터-모터 휠베이스)은 DJI 가 공개하지 않는다. 코드는 그 값을 추정으로
> 표시하고, **로터의 실제 위치는 따로 선언한다** — 275 mm 는 암 두께·모터 비례식의 스케일로만 남는다.

← 출처: `src/drones.py` mini5pro `note`. 지금 남아 있는 «모른다» 목록은 §4.4 에 모아 두었다.

### 2.3 ⭐ 참조 자료는 어디까지 제작에 들어갔나

«참조 자료는 채점에만 쓰고 제작에는 안 쓴다» 는 **깔끔하지만 지금은 사실이 아니다.**
참조 자료가 제작에 들어간 자리가 셋 있고, 그것을 감추면 «독립 채점» 이라는 말이 과장이 된다.

| # | 어디에 | 무엇이 들어갔나 | 등급 |
|---|---|---|---|
| ① | **matrice4e 형상 상수 14건** | DJI Matrice 4T 공식 STEP 의 모서리 실측 (접지 −59.82 · 갑판 crown 69.18 · RTK 꼭대기 89.70 · 셸 중심 x 41.71 mm 를 0.1 mm 안에서 재현) | **[A]** |
| ② | **mini2 전 형상 상수** | DJI 공식 GLB(WM161) 실측 — 셸 6 스테이션 중 가운데 4개가 GLB 와 0.5 % 안 | **[A]** |
| ③ | **프로펠러 평면형** (`BLADE_LAW_CANON="per_airframe"`) | 기체마다 **그 기체의 순정 프로펠러**를 실측·계측해 얻은 시위 분포. matrice4e 1157F · mavic4pro 1158F · mini5pro 6028F · mini2 4726F · s1000plus 1552 … | **칸마다 다르다** — [A] mini2 · [A−] typhoonh480 · [B] matrice4e·phantom3 · [B−] mavic4pro·s1000plus·m350rtk · [C] mini5pro·phantom4 · [D] x500v2 |

← 출처: ①은 `outputs/meshfix_matrice4e.json` + `outputs/mesh_inspect_body_arms_0816.json`
`meshfix_matrice4e_landed`(착지 검증) · ②는 `per_drone.mini2` ·
③은 `outputs/prop_law_by_airframe_0816.json` `C_law_by_airframe` + `src/geom.py` `BLADE_LAW_CANON`.

⭐ **③ 이 이번 판에서 가장 크게 달라진 자리다.** 옛 법칙(`BLADE_LAW=legacy`)은
**10기종 전부가 참조 프로펠러 하나(3DR Solo)의 날 모양**을 쓰는 판이었다.
정본에서는 기체마다 다르고, 그 «다름» 이 지어진 메쉬에서 실제로 확인된다 —
기체쌍 45개 중
**42쌍이 1 % 넘게 구별**되고,
같은 것으로 나온 3쌍은 표에 **대리로 선언된**
자리(phantom3=phantom4=x500v2)다 ← 출처: `outputs/prop_law_verify_0816.json` `V3_summary`.
기체별 표와 근거는 mesh05 가 맡는다.

**그래서 «독립 채점» 이라는 말을 어떻게 써야 하나.** 정확한 문장은 이렇다:

> 채점자 중 **실기체 3D 스캔(Phantom 4, CC-BY)** 은 제작에 한 번도 안 들어갔다 —
> 그 기체에 대해서는 채점이 진짜로 독립이다.
> 반대로 matrice4e·mini2 의 공식 CAD 는 **제작에 들어갔으므로**, 같은 CAD 로 다시 채점한
> 결과는 «맞췄다» 가 아니라 «반영이 착지했다» 로 읽어야 한다.

미리보기 하나: 우리 Phantom 4 메쉬는 실기체 스캔과 표면 거리 중앙값 **4.6 mm**,
90분위 11.7 mm 안에서 겹친다
← 출처: `report_mesh/outputs/mesh_verify_canon_0817.json` `G_scan.scan_to_cad_mm`.
이 기체는 제작에 스캔을 안 썼으므로 이 숫자는 독립 채점이다. 방법과 그림은 mesh08 에서.

## 3. 핵심 원칙 — "OBJ 1개 = 부위 1개 = Sionna 재질 1개"

드론 한 대를 한 덩어리 파일로 저장하지 않고, **부위마다 별도 OBJ 파일**로 저장한다.
이유는 Sionna 의 규칙 때문이다:

> Sionna 는 'OBJ 1개 = SceneObject 1개 = 재질 1개' 이므로, 부위별 재질을
> 주려면 이렇게 부위별로 나눠 저장한다.

← 출처: `src/geom.py` `write_obj_per_group()` docstring (그대로 인용);
같은 원칙이 `README.md` 에 "메쉬 원칙: OBJ 1개 = 부위 1개 = Sionna 재질 1개" 로 선언돼 있다.

함대 전체가 쓰는 부위 그룹은 **13개**다
(기체 하나가 그 전부를 쓰지는 않는다 — 열린 프레임 기체에는 셸이 없고, 접이식에는 데크가 없다):

| 부위(그룹) | 재질 키 | PO \|Γ\| | 쓰는 기체 | 함대 면적 [cm²] | 이 그룹이 무엇인가 |
|---|---|---|---|---|---|
| `body` | `plastic` | 0.28 | 9종 | 13,666 | 동체 셸 |
| `prop` | `prop_plastic` | 0.25 | 10종 | 6,026 | 프로펠러 |
| `battery` | `metal` | 1.00 | 10종 | 4,514 | 배터리팩(내부) — GHz 에서 파우치 포일은 사실상 금속 |
| `arm` | `carbon` | 0.90 | 3종 | 3,528 | 암 |
| `camera` | `camera_assembly` | 0.85 | 9종 | 3,385 | 짐벌 카메라(금속 하우징+유리렌즈) |
| `gear_cf` | `carbon` | 0.90 | 4종 | 2,774 | 카본 튜브 착륙장치 — 'gear'(플라스틱)와 재질이 다르다 |
| `gear` | `plastic` | 0.28 | 10종 | 2,369 | 착륙장치 |
| `pcb` | `pcb` | 0.80 | 10종 | 2,248 | ESC/메인보드(내부) — FR-4 + 구리 그라운드플레인 |
| `motor` | `metal` | 1.00 | 10종 | 2,125 | 모터 |
| `canopy` | `plastic` | 0.28 | 7종 | 1,434 | 상단 캐노피/배터리 |
| `deck` | `carbon` | 0.90 | 1종 | 1,212 | 카본 데크(상·하판 + 스탠드오프) — 셸 없는 열린 프레임 |
| `accent` ⚠ | `plastic` | 0.28 | 4종 | 1,163 | 전방 식별색 |
| `fc` | `pcb` | 0.80 | 1종 | 112 | 비행제어기(Pixhawk 류) — 상판 위 노출 |

← 출처: 재질 키·뜻은 `outputs/mesh_inspect_materials_check_0816.json` `assignment_audit`,
**면적은 정본 판 실측**(`report_mesh/outputs/mesh_canon_0817.json` `material_weighted.*.groups.*.area_mm2`, 10종 합계).
⚠ 표시는 «배정 자체가 다시 봐야 하는 칸» 이다 — §4.4 참조.

### 3.1 같은 재질인데 숫자가 둘이다 — 두 계산 경로가 다른 값을 쓴다

우리는 산란을 두 경로로 계산한다. **Sionna 재질**(광선엔진이 쓰는 (εr, σ) 슬래브)과
**PO 커널의 \|Γ\|**(면적분이 쓰는 실효 반사계수)다. 둘은 **일부러 다른 값**이고,
그 갈림이 얼마인지 적어 두는 것이 정직한 서술이다:

| 재질 키 | Sionna 경로 | Sionna \|Γ\| | PO \|Γ\| | 차이 [dB] | 왜 다른가 |
|---|---|---|---|---|---|
| `camera_assembly` | ITU metal | 0.99980 | 0.85000 | -1.41 | 짐벌 카메라 = **금속 하우징 + 유리 렌즈 + 짐벌 모터** |
| `pcb` | ITU metal | 0.99980 | 0.80000 | -1.94 | ESC/메인보드 = FR-4 유전체 + **구리 그라운드플레인** |
| `plastic` | custom (εr,σ) 슬래브 | 0.24369 | 0.28000 | +1.21 | 드론 셸(ABS/PC) |
| `plastic_blue` | custom (εr,σ) 슬래브 | 0.24369 | 0.28000 | +1.21 | 파란 모서리 트림 — 전파물성은 plastic 과 동일(색만 다름) |
| `prop_plastic` | custom (εr,σ) 슬래브 | 0.24369 | 0.25000 | +0.22 | 프로펠러 — **재질은 plastic 과 동일**(같은 ABS/PC·εr 2.7 → 렌더 색도 회색 동일) |
| `carbon` | custom (εr,σ) 슬래브 | 0.98867 | 0.90000 | -0.82 | 탄소섬유(도전성) — ITU 에 없음 |

← 출처: `outputs/mesh_inspect_materials_check_0816.json` `engine_divergence`
(fc = 3.5 GHz, 수직입사).

**왜 갈리나** — Sionna 쪽은 «반무한 벌크» 프레넬 값이고, PO 쪽은 «얇은 판의 앞뒷면 간섭까지
넣은 실효값» 이다. 드론 셸은 두께 **0.75 mm** 급이라 그 차이가 실재한다
← 출처: `docs/MATERIAL_CORRECTION.md`(셸 정본 두께 = DJI 공식 CAD 벽 두께 실측의 중앙값).

⚠ **현재의 한계** — 우리 PO 커널에는 **두께라는 개념이 없다.** \|Γ\| 하나를 상수로 받는다.
즉 PO 경로는 «두꺼운 판» 극한으로 계산하고, 두께는 그 상수를 고를 때 한 번만 반영된다.
이 한계는 지금 그대로 있고, 값은 발표까지 동결돼 있다.

### 3.2 «내부 금속이 없으면 속 빈 유령» — 방향은 맞고, 단서가 셋 있다

§2 의 문장을 지금 상태로 정확히 다시 쓴다.

**단서 ① — 우리 배터리는 상한값이다.** 팩 **외피 6면 전부**를 금속으로 둔다. 실물 팩은
플라스틱 케이스 안에 셀 스택이 들어 있어, 되쏘는 금속면은 더 작다. 크기는 1~3 dB 급이고
**미해결로 선언돼 있다** ← 출처: `outputs/mesh_inspect_internal_metal_0816.json` `battery_material`.

**단서 ② — 그 «내부» 금속이 실제로 셸 안에 있는 기체는 소수다.**

| 기체 | 판정 | 무엇을 뜻하나 |
|---|---|---|
| Mini 5 Pro | **FAIL** | 금속 상자의 일부가 셸 **밖**으로 나와 있다 |
| Mavic 4 Pro | **PASS** | 금속 상자가 전부 셸 안에 있다 |
| Matrice 4E | **FAIL** | 금속 상자의 일부가 셸 **밖**으로 나와 있다 |
| S1000+ | **N/A** | 설계상 열린 프레임이라 «셸 안» 이라는 물음이 성립하지 않는다 |
| Phantom 4 | **PASS** | 금속 상자가 전부 셸 안에 있다 |
| Typhoon H (H480) | **FAIL** | 금속 상자의 일부가 셸 **밖**으로 나와 있다 |
| X500 V2 | **N/A** | 설계상 열린 프레임이라 «셸 안» 이라는 물음이 성립하지 않는다 |
| Phantom 3 Professional | **FAIL** | 금속 상자의 일부가 셸 **밖**으로 나와 있다 |
| Matrice 350 RTK | **FAIL** | 금속 상자의 일부가 셸 **밖**으로 나와 있다 |
| Mini 2 | **FAIL** | 금속 상자의 일부가 셸 **밖**으로 나와 있다 |

← 출처: `report_mesh/outputs/mesh_canon_0817.json` `internal_metal`(정본 판에서 다시 잰 판정;
검사기는 `benchmark/mesh_internal_metal_check.py`).
이것이 왜 중요한가: 우리 SBR 경로는 **셸을 맞은 광선만** 내부를 투과로 본다. 금속 상자가
셸 밖으로 나와 있으면 그 상자는 «내부 산란체» 가 아니라 그냥 겉면이 된다.

**단서 ③ — 카메라 조립품의 \|Γ\|=0.85 는 출처가 없다.** 저장소가 스스로 그렇게 적는다
← 출처: `docs/MATERIAL_SOURCES.md` §6-4. 그 값을 유전체로 바꿔 보면 방위평균 σ 가
el 0/−30/−60° 에서는 −2.2…+0.1 dB 움직이는데, **바로 아래(나디르, el −90°)에서는
−6.5…+2.7 dB** 로 훨씬 크게 움직인다
← 출처: `outputs/mesh_inspect_gimbal_sensors_0816.json` `_summary.gate_D_dielectric_swing_db`.
원인은 짐벌을 매다는 **방진판이 수평 평판**이라 바로 아래 방향에 정반사가 서기 때문이다.

⇒ **절대 σ 를 인용할 때는 «배터리는 팩 외피 전체를 금속으로 본 상한값» 과**
**«카메라 0.85 는 출처 없는 값» 을 함께 적어야 한다.**

### 3.3 프로펠러 — 기체마다 그 기체의 날

정본 날 법칙은 `BLADE_LAW_CANON = "per_airframe"` 다 ← 출처: `src/geom.py`.
기체마다 **그 기체의 순정 프로펠러** 평면형(시위 분포)을 쓴다. 폭은 좁지 않다 —
c_max/R(날의 가장 넓은 곳 ÷ 반지름)이 기종에 따라 0.1756
~0.2687 로 **53 %** 벌어진다
← 출처: `outputs/prop_law_by_airframe_0816.json` `C_law_by_airframe`.

| 기체 | 정본 프로펠러 | c_max/R | 시위 정점 r/R | 근거 | 대리 |
|---|---|---|---|---|---|
| Mini 5 Pro | DJI 6028F | 0.2091 | 0.55 | **[C]** | — |
| Mavic 4 Pro | DJI 1158F | 0.1791 | 0.55 | **[B-]** | — |
| Matrice 4E | DJI 1157F (표준·순정 동봉) | 0.2004 | 0.30 | **[B]** | — |
| S1000+ | DJI 1552 / 1552R (거울쌍) | 0.1756 | 0.30 | **[B-]** | — |
| Phantom 4 | DJI 9450S (유력·미해결) | 0.2687 | 0.35 | **[C]** | ⛔ phantom3 대리 |
| Typhoon H (H480) | Yuneec Propeller A / B (YUNTYH118A / YUNTYH118B) | 0.1766 | 0.45 | **[A-]** | — |
| X500 V2 | 1045 (범용 규격 — Holybro X500 V2 킷 동봉) | 0.2687 | 0.35 | **[D]** | ⛔ phantom3 대리 |
| Phantom 3 Professional | DJI 9450 (자동조임) | 0.2687 | 0.35 | **[B]** | — |
| Matrice 350 RTK | DJI 2110s | 0.1809 | 0.25 | **[B-]** | — |
| Mini 2 | DJI 4726F | 0.2572 | 0.45 | **[A]** | — |

⛔ **[C]·[D] 칸은 그대로 읽을 것** — mini5pro·phantom4 는 계열 유추이고, x500v2 는 다른
기체(phantom3)의 프롭을 대신 세운 **대리**다. 그 세 칸의 c_max/R 은 «이 기체를 쟀다» 가 아니라
«이 기체는 이럴 것이다» 라는 뜻이다.

이 절은 자리를 잡을 뿐이고, 날 면적 변화·두께 축·검증 방법은 **mesh05** 가 맡는다.

지금 확실한 것만 함께 적는다.

- **움직이는 성분(AC)은 정의상 프로펠러만 남는다.** 정지한 동체의 기여는 순수 DC 라
  DC 를 걷어내면 사라진다. 이것은 측정이 아니라 구조적 필연이다.
- **총 반사(σ)로는 동체가 훨씬 세다** — matrice4e 부품 분해에서 프로펠러 몫은
  el −30° 에서 2.4 %(16.2 dB 아래), el 0° 에서 0.2 %(28.1 dB 아래)다.
- ⇒ «프로펠러가 표적을 지배한다» 는 **AC 채널 한정 문장**이다. 총 σ 문장으로 옮겨 쓰면 틀린다.

← 출처: 부품 분해는 `docs/MESH_AUDIT_0816.md` §④-1(우리 PO 커널로 matrice4e 를 부품별로 분해).

덤으로 **분절(articulated) 자세**도 부위별 OBJ 에서 공짜로 얻는다: 몸체와 프로펠러가 별개
조각이라 몸체 기울기와 로터별 회전 위상을 따로 줄 수 있다
← 출처: `src/drones.py` `pose_articulated()` docstring.

## 4. 전체 지도 — 자료 → 제작 → 검사기 → 원장

![pipeline map](outputs/figures/pipeline_map.png)

**그림 3** — 모든 정보가 어디서 와서 어떻게 채점되는지 한 장 지도
← 그림 생성: `report_mesh/src/viz_mesh_reports.py` `fig_pipeline_map()`.

층이 **넷**이다. 앞의 세 층은 예전부터 있었고, 넷째(원장)를 따로 세운 것이 지금 구조다.

**① 자료층** — 세 종류, 역할이 다르다:

- **제조사 공식 제원표** → `docs/drone_research.json` → `docs/SPECS.md`. **모든 기체의 치수 출발점.**
- **제조사 공식 CAD** (Matrice 4T STEP · Mini 2 GLB · X500 v2 STEP) — **제작에도 들어간다**(§2.3).
- **실기체 3D 스캔 / 타사 실물 CAD** — 채점 전용.

**② 제작층** — 숫자가 형상이 되는 곳:

- `src/drones.py` 의 `DroneSpec` 10개 → `src/drone_cad.py` + `src/cadkit.py` 가
  드론을 **trimesh+manifold3d (drone_cad)** 엔진으로 깎는다(드론 제작 경로는 이 하나뿐이다)
  ← 출처: `report_mesh/outputs/mesh_verify_canon_0817.json` `_meta.mesh_engine`.
- **왜 trimesh + manifold3d 인가?** 프리미티브를 그냥 겹쳐 놓으면 겹친 파트의 **내부에 숨은 면**이
  표면 데이터에 그대로 남고, PO 는 그런 면까지 반사면으로 센다. manifold3d 의 **불리언 합집합**은
  겹친 파트를 한 덩어리로 녹여 내부 면이 애초에 존재하지 않게 한다
  ← 출처: `src/drone_cad.py` 머리말 "왜 이게 RCS 에 중요한가".
  (자작 `geom.Mesh` 의 역할은 **컨테이너와 무대**다: 완성 메쉬 담기(.v/.f/.g)·부위별 OBJ 저장,
  그리고 범용 프리미티브 제작.)
- 완성 메쉬는 `write_obj_per_group()` 으로 **부위별 OBJ** 저장(§3) → `src/materials.py` 가
  부위→전파재질 배정.

**③ 검사기층 · ④ 원장층** — §4.2~§4.3. 그 전에 «어느 판을 짓고 있나» 부터(§4.1).

### 4.1 ⭐ 지금 기본으로 켜져 있는 것 — 정본 판

제작층에는 **판(version)을 고르는 스위치가 둘** 있다. 둘 다 `src/geom.py` 한 곳에 있고,
아무것도 안 주면 **정본**이 켜진다. 지금 이 리포트의 모든 수는 그 정본 판에서 잰 것이다.

| 스위치 | 지금 기본값(정본) | 무엇이 달라지나 | 옛 판으로 되돌리는 법 |
|---|---|---|---|
| `geom.MESH_FIX_CANON` | `battery, i5` | `battery` = 배터리 팩 상자와 구조판 상자가 서로 파고든 것을 불리언 합집합으로 없앤다(4기체) · `i5` = mini2 셸의 구멍을 닫는다 | `MESH_FIX=none` |
| `geom.BLADE_LAW_CANON` | `per_airframe` | 기체마다 **그 기체의 순정 프로펠러** 평면형을 쓴다 | `BLADE_LAW=legacy` |
| 파일명 꼬리표 | `_mfixbatteryi5_blperairframe` | 정본 판 산출물의 이름에 붙는다 | 옛 판은 꼬리표가 **없다** — 그래서 두 판이 이름만으로 갈린다 |

← 출처: `src/geom.py` `MESH_FIX_CANON`·`BLADE_LAW_CANON` 선언.

**두 수리가 각각 무엇을 고치나** (`MESH_FIX_CANON`):

- **`battery`** — 배터리 팩 상자와 그 아래 구조판 상자를 **불리언 합집합**으로 한 덩어리로 녹인다.
  두 상자를 그냥 겹쳐 놓으면 겹친 자리의 면이 껍질 **안쪽**에 남고, 우리 PO 커널은 가림을
  안 보므로 그 면적을 두 번 센다(그 겹침이 4기체에서 47.9~50.0 %다 — mini2·phantom4·
  mavic4pro·mini5pro. 기종별 실측 부품표가 있는 matrice4e·phantom3 은 0 % 라 안 바뀐다).
  **치수는 하나도 안 바꾼다.**
  σ 로는 방위평균 **+1.02~+1.88 dB**(최악 방위 9.6~18.9 dB) — 2층 항목 중 가장 크다.
  ⚠ 부피가 **−12.27 %** 줄어 질량 −21~31 g · 관성 대각 +1.9~2.8 % 가 따라 움직인다.
  로터 요동(자세 응답) 모델을 다시 잴 때 이 값을 쓸 것.
- **`i5`** — 불리언 합집합이 남기는 바늘 삼각형을 **모서리 붕괴**(짧은 변의 두 끝점을 하나로
  합쳐 삼각형을 접는 것)로 없앤다. 그냥 지우면 그 자리에 테두리가 남아 mini2 셸에 경계 모서리
  3개짜리 구멍이 생긴다.
  **σ 로는 없다**(1e-10 dB). 켜는 이유는 산란이 아니라 **검사 정확도**다 — 껍질이 안 닫히면
  `contains()`(안/밖 판정)가 성립하지 않아 검사가 그 부품을 컨테이너에서 빼 버린다.
  그 상태에서 매몰면을 재면 **29.71 %** 로 읽히는데 참값은 **44.45 %** 다 — **14.74 pp** 를 못 본다
  ← 출처: `outputs/mesh_layer2_holes_poles_0816.json` `I5_mini2_body_구멍`.

⭐ 이 두 가지가 «켜도 되는 것» 으로 분류된 기준은 하나다 — **σ 영향이 판정 밴드 밖이고,**
**근거가 저장소 안에 이미 있다.** 그 기준을 못 넘긴 수리들(`i3` 매몰면·`i4` 캐노피·`m4`·`m6`)은
켜지 않고, 인증서의 «장담 못 함» 목록에 영구 한계로 올려 두었다(§4.5)
← 출처: `src/geom.py` `MESH_FIX_CANON` 주석 A통/B통.

⭐ **판정은 호출 시점에 한다.** `geom.mesh_fix_set()`·`geom.blade_law_canon()` 이 환경변수를 **부를 때마다** 읽으므로, import 뒤에 켜도 듣는다 ← 출처: `src/geom.py` 두 함수의 docstring.

⭐ **꼬리표가 규약인 이유** — 정본 판 산출물은 이름에 `_mfixbatteryi5_blperairframe` 가 붙는다. 이름이 같으면 계산기가 **옛 판 결과를 재사용**하고, 재계산이 «건너뜀» 으로 끝나 버린다 ← 출처: `benchmark/elevation_sweep_md.py` 꼬리표 블록.

⛔ `MESH_FIX=none BLADE_LAW=legacy` 를 주면 옛 판이 **비트동일**하게 다시 나온다 ← 출처: `benchmark/regress_blade_law_bitidentical.py` · `src/mesh_check.py` legacy 회귀. 전환 직전 산출물은 `/data/public/sionna/archive_pre_meshfix_20260817/` 에 있다(꼬리표 없는 샤드 3,813개 + README).

### 4.2 검사기는 하나가 아니다 — 7개고 역할이 다르다

| 검사기 | 언제 도나 | 무엇을 보나 | 범위·단서 |
|---|---|---|---|
| `src/cadkit.py` `Assembly.check` | 빌드 도중 | 파트 하나를 붙일 때마다 | 부품 단위 수밀·법선 |
| `src/mesh_check.py` | 출하 게이트 | 11 검사 + 예산표 | `python src/drones.py`(OBJ 내보내기) 한 문에 배선. `MESH_GATE=off` 로 끌 수 있고, RCS·렌더가 쓰는 **인메모리 `build_drone()` 은 이 문을 안 지난다** |
| `report_mesh/src/verify_mesh_suite.py` | 원장 생성 | A~I 9절 | 이 시리즈의 숫자를 만든다. I 절(SBR)만 GPU |
| `report_mesh/src/verify_mesh_canon_0817.py` | 원장 생성(정본) | A·B·C·D·F·G | 같은 잣대(위 스위트의 `sec_*`)로 **정본 판**을 다시 잰다. 전부 CPU |
| `benchmark/check_gimbal_sensors_0816.py` | 특수 검사 | 짐벌·센서 게이트 A~D | 부착·삼킴·선언초과·재질 민감도 |
| `benchmark/mesh_internal_metal_check.py` | 특수 검사 | 내부 금속 포함 판정 | «금속 상자가 정말 셸 안인가» |
| `benchmark/mesh_certify.py` | ⭐골든 봉인 | 형상·치수·예산·바깥참값·문·인증서 여섯 축 | «오늘이 어제와 같은가» 를 지킨다. 봉인 대조 약 9 초 · `--full` 약 295 초. ⚠지키는 것은 «안 바뀜» 이지 «옳음» 이 아니다 |

← 출처: 각 파일의 모듈 docstring·`__main__` 배선.

**게이트의 범위를 정확히 적는다.** `src/mesh_check.py` 의 회귀 게이트는
`python src/drones.py`(부위별 OBJ 내보내기) **한 문**에 걸려 있다. 세 가지 단서가 있다:

1. 환경변수 `MESH_GATE=off` 로 끌 수 있다.
2. RCS·렌더·마이크로도플러가 쓰는 **인메모리 `build_drone()` 은 이 문을 안 지난다** —
   전 기종 검사가 수십 초 걸려서 import 시점에 걸지 않는다고 코드가 스스로 적는다.
3. 그래서 «메쉬를 쓰는 모든 경로가 검사를 통과한다» 고 쓰면 지금 상태보다 강한 말이 된다.

**원장층(④)** — 검사 결과가 모이는 파일들이다. 이 편 머리의 «무엇을 근거로 하는가» 표가
그 목록이고, 이 노트북의 모든 숫자가 거기서 나왔다.

### 4.3 무엇을 검사하나 — 11 검사, 그리고 «예산» 이라는 규약

| # | 검사 | 무엇을 묻나 |
|---|---|---|
| 1 | **수밀(watertight)** | 부품이 닫힌 껍질인가 |
| 2 | **경계 모서리** | 삼각형 하나만 쓰는 모서리 = 구멍의 테두리. **원칙은 0**, 예산으로만 예외 |
| 3 | **winding** | 이웃한 면이 같은 방향으로 감겼는가 |
| 4 | **법선 방향** | 닫힌 부품의 부호있는 부피가 양수인가 |
| 5 | **부호부피(원본 인덱스)** | trimesh 를 **전혀 안 거치고** 출하 인덱스에서 손계산 |
| 6 | **퇴화면 — 절대 + 상대** | 면적 잣대에 더해 **최소 내각 <0.5°** 슬리버를 센다 |
| 7 | **그룹 안 겹침** | 같은 그룹의 두 부품이 서로 파묻혔는가(PO 면적 이중계상) |
| 8 | **치수 대조** | 프롭 지름·로터 대각·공표 외형을 `DroneSpec` 의 수와 대조 |
| 9 | **손대칭성** | 로터별 날 비틀림 방향이 회전방향과 맞는가 — **거울상 기체 탐지** |
| 10 | **프롭↔모터 벨 관통** | 원통 근사(빠름) + 솔리드 내부판정(판정 기준) |
| 11 | **⭐ 매몰면 전수** | **전 부품쌍**에서 다른 부품 솔리드 안에 든 면적. «설계 의도»(셸 안의 배터리·기판)와 «진짜 결함» 을 갈라 뒤쪽만 예산에 건다 — `check_buried_faces` · `BURIED_FACE_BUDGET_PCT` |

← 출처: `src/mesh_check.py` 모듈 docstring·구현.

⭐ **«통과» 의 뜻이 «0» 이 아니다.** 검사기는 항목마다 **예산 표**를 들고 있고,
통과란 «선언된 예산 안» 이라는 뜻이다:

| 예산 | 지금 걸려 있는 값 | 무엇을 뜻하나 |
|---|---|---|
| `BOUNDARY_EDGE_BUDGET_FIXED` | 기본 **0** — 예외 없음 | 구멍은 원칙적으로 없어야 한다. 정본에서는 예외 칸이 비어 있다 |
| `SLIVER_BUDGET_BLADE_LAW` | 6기체에 별도 값 — 실측 241~644 / 예산 266~709 (나머지 기체는 기존 `SLIVER_BUDGET` 실측 260~389) | 아주 뾰족한 삼각형 개수. 면적 비중이 0.0001~0.03 % 라 σ 에는 무해하고, 감시하는 이유는 법선이 수치적으로 불안정한데 PO 조명 판정이 `n̂·û>0` 이기 때문이다 |
| `PROP_BELL_SOLID_AREA_PCT_BLADE_LAW` | typhoonh480 6.5 % · m350rtk 5.4 % | 프로펠러가 모터 벨 솔리드 **안**에 든 면적 비율 |
| `GROUP_OVERLAP_BUDGET_FIXED` | 기본 0.1 % (battery 실측 0.0 %) | 같은 그룹 안에서 부품이 파묻힌 비율 |
| `BURIED_FACE_BUDGET_PCT` | 기종별 7.6~37.7 % (실측 7.2~26.3 %) | 다른 부품 솔리드 **안**에 든 면적 중 «진짜 결함» 몫 |
| `DIM_TOL_PCT` | 프롭 지름 1 % · 외형 1 % · 대각 3 % (mini5pro 예외 12 %) | 공표 숫자와의 허용 오차 |
| `HANDEDNESS_MIN_ABS` | 0.05 | 날 비틀림 지표의 최소 크기. 이보다 작으면 «비틀리지 않았다» 는 뜻이라 부호를 믿을 수 없다 |

← 출처: `src/mesh_check.py` 예산 표 · 실측은 `report_mesh/outputs/mesh_verify_canon_0817.json` `budget_usage`·`buried_faces`.

**예산 표를 왜 이렇게 쓰나** — 이 표들은 «이만큼이 옳다» 가 아니라 **«지금 이만큼이다» 라는**
**선언**이다. 값은 전수 실측으로 채웠고 여유는 약 10 % 다. 그래서 **새로 생기는 결함은**
**예산을 넘겨 실패한다.** 숨기지 않으면서도 회귀를 막는 방식이다.

⭐ **예산이 «법칙별» 로 갈렸다.** 옛 표(`SLIVER_BUDGET`·`PROP_BELL_SOLID_AREA_PCT`)는 전부 옛 날 법칙(`legacy`)에서 잰 스냅샷이다. 정본은 기체마다 다른 평면형으로 로프트를 다시 뜨므로 씨접합 슬리버 수와 뿌리 겹침이 달라진다 — 결함이 는 것이 아니라 **다른 형상**이다. 그래서 표를 덮어쓰지 않고 **법칙을 키에 넣어** 따로 선언한다(`SLIVER_BUDGET_BLADE_LAW` 6행 · `PROP_BELL_SOLID_AREA_PCT_BLADE_LAW` 2행).

⭐ **값은 실측 + 10 % 로만 두고 실측치를 괄호에 남긴다.** «예산을 올려 통과시킨다» 는 인증서가 이름 붙인 안티패턴이라, 얼마를 올렸는지 소스에서 바로 읽히게 한다 ← 출처: `src/mesh_check.py` 두 표의 주석 · `docs/MESH_CERTIFICATE.md` §1-③.

**이 검사가 아직 못 보는 것** — 정직하게 남긴다:

- **동일평면 겹침** — 두 부품 표면이 정확히 같은 자리에 있으면 관통 검사가 못 본다. 9기체에서 34쌍이 그 상태다(가장 큰 것은 s1000plus body↔battery 16,745.8 mm²).
- **기종별 재질 분기** — `drone_gamma_map(spec, fc)` 이 `spec` 을 안 쓴다. 지금은 재질이 기체와 무관해서 맞지만, 기종별 재질이 생기는 순간 조용히 틀린 답을 준다.
- **PO 경로의 가림** — `rcs_po.py` 가 자기 docstring 에서 자기차폐·다중반사를 무시한다고 선언한다. 부품 속에 묻힌 면이 그 경로에서는 이중계상된다(재질 가중으로 +0.03~+0.98 dB · 가장 작은 것 s1000plus · 가장 큰 것 mini2).
- ⭐**출하한 파일 자체** — 검사는 메모리 배열에서 돌고 파일은 그 뒤에 쓰인다. 되읽어 같은 검사를 먹이면 10기체 중 2기체가 실패한다(1 µm 격자 반올림) ← 출처: `docs/MESH_CERTIFICATE.md` §1-①.
- ⭐**부품이 있는가** — 그룹이 통째로 사라져도 검사 10계열과 바깥 참값이 전부 조용하다 ← 출처: `docs/MESH_CERTIFICATE.md` §3.2.

## 4.4 지금 남은 결함 — 있는 그대로

아래는 **현재 메쉬가 안고 있는 어긋남**이다. 크기를 함께 적어, 어느 결론이 흔들리고
어느 결론이 안 흔들리는지 독자가 직접 판단할 수 있게 한다.

| 무엇 | 기체 | 지금 이만큼 | 어디에 실리나 |
|---|---|---|---|
| 공표 높이를 **형상이 아니라 세로 배율**로 맞춘다 | mini5pro · mavic4pro | 세로 배율 1.2985 / 1.3524 — 형상표의 셸 높이 45.05 / 62.10 mm 가 메쉬에서 59.99 / 87.70 mm 로 나온다 | 평판극한 σ 상한 +2.27 / +2.62 dB (방위평균, el 0°) |
| 짐벌이 착륙발보다 아래 | mavic4pro | 카메라 최저점이 발보다 15.35 mm 아래. 발을 바닥으로 놓고 같은 규칙을 풀면 세로 배율이 1.3524 → 1.5977 (18.14 %) | 위 세로 배율의 **원인** — 예산 구멍을 가린다 |
| 뜬 파트(기체에 안 닿는 부품) | phantom4 · phantom3 · m350rtk · x500v2 | 착륙아치 8.3~8.5 / 13.7~13.8 mm · 프롭 허브 6.0 mm · 레일 4.0 mm | 간극 0.05~0.16 λ @3.5 GHz — 면적은 그대로고 가림·다중반사·위상이 바뀐다 |
| 로터면이 공식 CAD 보다 위 | matrice4e | 18.5 mm = 0.216 λ @3.5 GHz. 명세가 F19~F21 로 «엔진 변경 필요» 라 미뤄 둔 자리 | 프롭 장착 높이가 함께 움직인다 |
| ⭐**바깥 참값(실물 치수)과 어긋나는 행** | 함대 | 77행 중 **24행**. 가장 큰 것 — m350rtk 펼침 폭 −6.5 % · 길이 −4.8 % · mini5pro 배터리 높이 +29.5 % · 길이 −19.9 % | «실물과 같은가» 축. 인증서가 **장담 못 한다**고 선언한 자리다 |
| 짐벌이 세 조각으로 떨어져 있다 | phantom3 | 방진판·요 샤프트·카메라 블록이 서로 2.99~8.18 mm 씩 벌어져 있고, 기체 표면과도 2.44~37.67 mm 떨어져 있다 — 잇는 구조가 없다 | 면적 360 cm² 가 공중에 뜬다. 가림·다중반사가 달라지고, PO 와 SBR 이 같은 메쉬를 다르게 읽는다 |
| 짐벌 헬퍼의 «선언 치수» ↔ 실제 크기 | mini5pro·matrice4e·phantom4·s1000plus 등 | 인자로 넣은 상자 위에 요크·렌즈·마운트가 더 붙어 최대 1.57 배로 지어진다 | 인자를 실물 치수로 인용하면 틀린다. mini2 처럼 **역산**해야 실물과 맞는다 |
| 카본 판이 `plastic` 그룹에 있다 | s1000plus | 판 2장만 세도 body 합집합 전 면적의 69.3 % (스탠드오프 기둥까지 넣은 스택은 74.5 %) | 면 반사율 +10.14 dB (carbon 0.90 ↔ plastic 0.28) |
| 짐벌을 구속하는 바깥 검사가 없다 | 함대(특히 matrice4e) | 짐벌 폭은 사진 계측 대비 +26 % 인데 판정이 **면제**돼 있다(참값이 사진뿐이고 공식 CAD 는 4T 판이라 짐벌이 다른 물건이다) | 짐벌은 산란 기여가 큰 부품이라 이 빈칸은 작지 않다 |

← 출처: `outputs/mesh_inspect_body_arms_0816.json` `findings`·
`outputs/mesh_inspect_gimbal_sensors_0816.json` `_summary`·
`outputs/mesh_inspect_materials_check_0816.json` `findings`·
`outputs/mesh_cert_dimension_external_0816.json` `summary_by_airframe`.

**dB 를 읽는 법** — 위 표의 dB 는 대부분 **평판극한 상한**이다. «같은 크기 평판이라면 최대
이만큼» 이라는 뜻이지 커널이 계산한 σ 가 아니다. 크기 감각을 주는 자로만 쓸 것.

### 4.5 ⭐ 무엇을 장담하고 무엇은 못 하나 — 메쉬 인증서

위 결함 지도는 «지금 어긋난 자리» 다. 그것과 별개로, **검사 체계 자체가 무엇을 보증하는가** 를
따로 심사한 문서가 있다 ← 출처: `docs/MESH_CERTIFICATE.md`.

판정은 ⭐**«조건부 장담»** 이다 ← 출처: `docs/MESH_CERTIFICATE.md` §0.

| 무엇 | 지금 값 |
|---|---|
| 결함이 있는 자리의 **범주 지도** | 20개 (`M0`~`M19`) |
| 인증 매트릭스 | 450칸 — 통과 425 · **실패 0** · 어긋남 10 · 사각지대 1 · 해당없음 4 · 빈칸 10 |
| 적대 대조(일부러 결함을 심어 «무는가» 를 본다) | **184/184**, 6스위트 |
| 양성 대조가 걸린 검사 | 검사 45개 중 매트릭스 기준 **35개**. 인증서 라운드가 넷(A8·A9·G2·R0)을 더 심어 **남은 빈칸은 여섯**(A7·V7·Z1·Z3·G1·W1) |
| 골든 봉인 ↔ 지금 메쉬 | 형상 **10기체 전부 동일**(골든 지문). 별도로 인증서 4종 지문이 기체마다 맞는지도 본다 — **40/40** |
| 바깥 참값(실물 치수) | 77행 — 일치 50 · **어긋남 24** |
| 근거 등급이 붙은 칸 | 기체×부품 120칸 중 **37칸**. 그중 **독립 근거가 있는 칸은 15칸**이다(나머지 22칸은 독립 참값 행이 0개). ⚠ 별개 잣대인 «참값 행» 쪽에서 독립인 것은 77행 중 32행 — **칸 수와 행 수를 섞어 읽지 말 것** |

⇒ 한 문장으로: **자기 무결성(메쉬가 스스로 앞뒤가 맞는가)과 회귀(어제와 같은가)는 장담한다.**
**실물 충실도(실제 기체와 같은가)와 완전성(있어야 할 것이 다 있는가)은 장담하지 못한다.**

**⛔ 인증서가 «장담 못 한다» 고 선언한 것 — 그대로 옮긴다:**

- **출하한 파일 자체** — 검사는 메모리 배열에서 돌고 파일은 그 뒤에 쓰인다. `geom.Mesh.write_obj` 가 1 µm 격자로 반올림하므로, 되읽어 같은 검사를 먹이면 **10기체 중 2기체(matrice4e·mini2)가 실패한다**(면적 0 삼각형 6장 · 비다양체 모서리 6개). 물리 크기는 무시할 만하지만 «퇴화면 0 장» 은 출하물에 대해 참이 아니다.
- **부품이 있는가** — 그룹이 통째로 사라져도 검사 10계열과 바깥 참값이 전부 조용하다(canopy 표면적 8.4 % · gear 2.1 % 를 지워 확인).
- **매몰면 예산의 자기참조** — 잣대가 전체 표면적 대비 비율이라, 부품 하나를 통째로 묻어도 예산을 못 넘고 값이 거꾸로 가기도 한다. 지금 그 검사가 하는 일은 «오늘보다 나빠지지 않았다» 의 감시뿐이다.
- **실물 충실도** — 바깥 참값 77행 중 **24행이 어긋나 있다**(가장 큰 것: m350rtk 펼침 폭 −6.5 % · mini5pro 배터리 높이 +29.5 %).
- **독립 참값이 0행인 기체가 둘** — s1000plus · m350rtk. 이 둘의 «치수가 맞다» 는 **우리 상수를 우리가 다시 읽은 것**이다.
- **재질 물성값(εr·tanδ)의 옳음 · 커널(PO·SBR)의 옳음** — 인증서가 스스로 «이 인증서의 범위 밖» 이라고 선언한다.

### 지금 «모른다» 고 선언한 것

- mavic4pro 의 세로 예산 35.2 mm 가 **어디에** 있어야 하는지 못 정했다. 공표 언폴드 높이 135.2 mm 는 공식이지만 그 135.2 를 다리·셸·짐벌·모터에 어떻게 나누는지는 사진 한 장으로 안 풀린다 — matrice4e 처럼 공식 CAD 가 필요하고 DJI 는 Mavic 4 Pro CAD 를 공개하지 않는다.
- mini5pro 셸 높이의 1차 출처가 없다. `fh = 0.495` 는 공표 높이(91, **프롭 포함**)에 대한 비율이고, 그 91 자체가 프롭을 포함하므로 셸 높이를 직접 구속하지 않는다. 폴디드 68 mm 로 교차검산하려면 짐벌 매달림 길이를 따로 재야 하는데 그 값도 실측이 없다.
- B3 의 «기수 정면 정반사 10 dB» 는 평판극한 상한일 뿐 커널 결과가 아니다. 진짜 값을 알려면 스무딩 0/4 두 메쉬로 PO 를 돌려야 하는데 이 라운드는 σ 파일을 열지 않았다.
- phantom3·phantom4 착륙아치가 «어디에» 붙어야 하는지 — 매뉴얼 정면도가 붙는 곳 좌우 스팬은 주지만 앞뒤 부착점은 셸 곡면과의 교선이라 표에서 못 읽는다.
- x500v2 배터리 트레이 2.65 mm 는 2026-08-04 원장이 «면-대-면 접촉의 거짓양성» 이라 적었는데 양방향 잣대로도 남는다. 어느 쪽이 맞는지 판정하지 않았다.
- **배터리 재질** — ⚠ **미해결로 선언한다.** 지금 고치지 않는 이유: 셀 스택의 실제 치수가 1차 출처 0 이고, 추정으로 줄이면 «측정 아닌 값» 을 또 하나 심는다. 대신 **모든 절대 σ 인용에 «배터리는 팩 외피 전체를 금속으로 본 값(상한 쪽 1~3 dB)» 단서를 붙일 것.**
- **카메라 재질 0.85 의 출처** — docs/MATERIAL_SOURCES.md §6-4 가 이미 «출처 없음 · 총 σ 를 최대 1.81 dB 움직임» 으로 적어 뒀다 (그 1.81 은 mavic4pro·1.843 GHz 한 조건에서 잰 값이다).
- **프로펠러 날 두께** — 사진으로는 **원리적으로** 못 잰다(겉보기 높이가 시위·피치각에 지배되고 두께 항은 그 5분의 1 아래다). 실제로 잰 기체는 mini2 하나뿐이고, typhoonh480 은 두께비만 있다. 나머지 여덟은 공용 상수를 쓴다. ⚠ 평면형(시위 분포)은 기체별로 **닫혔다** — 안 닫힌 것은 두께 축뿐이다.
- **분절(움직이는) 메쉬 열의 정확도** — 인증서가 자기 범위 밖으로 선언한다(«자세 재현» 만 보고 양성 대조가 없다).

⭐ **빈칸이 가짜 값보다 낫다.** 위 항목들은 값을 채워 넣는 대신 비워 두었다.

## 5. 시리즈 목차 — mesh02~08

| 편 | 주제 (한 줄) |
|---|---|
| **mesh02** | 도구 상자 — 어떤 파이썬 라이브러리를 왜 골랐나, 그리고 검사기는 무엇을 보고 무엇을 못 보나 |
| **mesh03** | 자료 수집 — 모든 숫자·모델의 출처(공식 제원·공식 CAD·실기체 스캔·타사 CAD, 라이선스) |
| **mesh04** | 몸체 CAD — 스펙 숫자가 드론 모양이 되기까지(로프트·조립 순서·기종별 개성) |
| **mesh05** | 프로펠러 — 기체마다 그 기체의 순정 날(정본 법칙 `per_airframe`), 그 근거와 검증 |
| **mesh06** | 색이 곧 재질 — 부위별 전파 재질과 두 계산 경로의 \|Γ\| |
| **mesh07** | 검증 ① 기하 — 수밀·법선·삼각형 품질·대칭·부위 겹침 |
| **mesh08** | 검증 ② 실물·물리 — 치수 대조·실기체 스캔 chamfer·수치 수렴 |

모든 편이 이 편과 같은 규약을 따른다: 생성물 노트북, 수치는 원장에서 주입, 사실마다 `← 출처:`.

**이 시리즈의 지위** — `README.md` 편성에서 report_mesh 8편은 **부록**이다. 본편 쪽에는
같은 주제의 별편 **2-3 «표적을 짓는다 — 메쉬와 재질»**(`reports/02_3_target-mesh.ipynb`)이
따로 있다. 둘의 차이는 독자다 — 별편 2-3 은 결론을 쓰고, 이 시리즈는 **만드는 법과 채점 방법**을 쓴다.

## 재현 명령

```bash
PY=/workspace/.venvs/py312/bin/python
cd /workspace/sionna

# 0) 지금 어느 판인가 — 아무것도 안 주면 정본이다
PYTHONPATH=src $PY -c "import geom; print(geom.mesh_fix_set(), geom.blade_law_canon())"

# 1) 정본 원장 재생성 — 이것부터. 전부 CPU.
PYTHONPATH=src:benchmark $PY report_mesh/src/verify_mesh_canon_0817.py   # A·B·C·D·F·G
PYTHONPATH=src:benchmark $PY report_mesh/src/mesh_canon_0817.py         # 스위치·예산·매몰

# 1b) H·I 절(PO 수렴·SBR)은 아직 옛 원장에만 있다. I 절은 GPU 를 쓴다.
PYTHONPATH=src:benchmark $PY report_mesh/src/verify_mesh_suite.py             # 전체
PYTHONPATH=src:benchmark $PY report_mesh/src/verify_mesh_suite.py --skip-sbr  # GPU 없이 A~H

# 2) 그림 재생성
PYTHONPATH=src:benchmark $PY report_mesh/src/viz_mesh_reports.py

# 3) 노트북 재생성
PYTHONPATH=src:benchmark $PY report_mesh/src/make_mesh01.py

# 4) 회귀 봉인 — 형상이 그대로인가 (약 9 초)
PYTHONPATH=src:benchmark $PY benchmark/mesh_certify.py

# 5) 옛 판을 되살릴 때 (비트동일) — 산출물 이름에 꼬리표가 안 붙는다
MESH_FIX=none BLADE_LAW=legacy PYTHONPATH=src:benchmark $PY <스크립트>
```

⭐ **어느 판에서 잰 원장인지 확인할 것.** 정본 원장은 머리에 꼬리표 `_mfixbatteryi5_blperairframe` 를 적어 둔다
(`report_mesh/outputs/mesh_verify_canon_0817.json` `_meta.file_tag`). 두 원장의 꼬리표가 다르면 이 노트북의 생성기가
**일부러 멈춘다** — 세대가 섞인 표를 조용히 찍는 것보다 낫다는 규약이다.

⭐ **순서를 지켜야 한다.** 생성기는 원장이 레지스트리와 다르면 **일부러 멈춘다** —
«원장 10종 vs 레지스트리 N종» 으로 예외를 던진다
← 출처: `report_mesh/src/mesh_ledger.py` `ledger_order()`.
리포트가 틀린 개수를 조용히 쓰는 것보다 멈추는 편이 낫다는 규약이다.


⚠ **지금 원장의 I 절(SBR)은 이월된 값이다** — `--skip-sbr` 로 돌린 갱신이 GPU 증거를
지우지 않게 직전 원장에서 옮겨 왔고, `stale: true` 로 표시돼 있다. 다른 절과 세대가
다르므로 면 수·σ 를 나란히 인용하지 말 것.

---

**다음 편** → [mesh02 — 도구 상자](mesh02_tools.ipynb) : 어떤 파이썬 라이브러리를 왜 골랐고,
검사기가 무엇을 보고 무엇을 못 보는지.